# External Shock Propogation into Polymarket

This notebook is a fast feasibility study for the shock-propagation idea.

The question is simple: if a liquid external market such as `BTC/USD` or `ETH/USD` experiences a large shock, do Polymarket markets reprice afterwards in a systematic way?

This is not yet a final model notebook. It is an empirical screen for whether the idea has signal worth pursuing.


## Setup

We will:

1. detect external shocks in `BTC/USD` and `ETH/USD`
2. attach those shocks to Polymarket repricing snapshots
3. compare repricing rates and move magnitudes under shock vs non-shock states
4. inspect concrete examples of markets that reacted after shocks

The intended interpretation is one of two things:

- delayed information uptake from more liquid markets into event markets
- decomposition of common external volatility into event-specific risks


## Environment and Imports

This cell imports the benchmark helpers and external-covariate utilities. Reusing the benchmark code keeps the experiment aligned with the repricing setup used elsewhere in the repository.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from polymarket_research import PolymarketDataset
from polymarket_research.benchmarks.covariate_utils import (
    asof_join_covariates,
    load_external_covariates,
    pivot_covariates_to_wide,
)
from polymarket_research.benchmarks.dataset_utils import (
    load_snapshot_frame,
    prepare_resolved_markets,
)
from polymarket_research.utils import setup_root

REPO_ROOT = setup_root()


## Configuration and Helpers

This cell fixes the domains, the repricing horizon, and the shock definition.

The shock detector is intentionally simple: we compute returns, scale them by a rolling volatility estimate, and flag `|z| >= 2`. For a feasibility study this is enough; if the idea looks promising, a stronger version can try multiple horizons and more careful event definitions.


In [ ]:
DB_PATH = DEFAULT_DB_PATH
DOMAINS = ('crypto', 'politics', 'geopolitics', 'technology', 'finance_economy')
MAX_MARKETS_PER_DOMAIN = 120
MIN_PROBABILITY_ROWS = 288

REPRICING_FUTURE_HOURS = 24
REPRICING_LOOKBACK_HOURS = 24
REPRICING_SAMPLE_EVERY_HOURS = 12
REPRICING_MOVE_THRESHOLD = 0.15

SHOCK_Z_THRESHOLD = 2.0
SHOCK_STD_WINDOW = 288
EXTERNAL_PATH = REPO_ROOT / 'cached_data' / 'external_covariates'


def build_shock_table(path: Path, z_threshold: float = 2.0, std_window: int = 288) -> pd.DataFrame:
    covariates = load_external_covariates(path)
    wide = pivot_covariates_to_wide(covariates, value_col='value').sort_values('timestamp_utc').reset_index(drop=True)
    out = wide[['timestamp_utc']].copy()
    value_cols = [col for col in wide.columns if col != 'timestamp_utc']
    for col in value_cols:
        series = pd.to_numeric(wide[col], errors='coerce')
        ret = series.pct_change()
        sigma = ret.rolling(std_window, min_periods=max(24, std_window // 6)).std()
        z = ret / sigma.replace(0.0, np.nan)
        out[f'{col}_level'] = series
        out[f'{col}_ret'] = ret
        out[f'{col}_z'] = z
        out[f'{col}_shock'] = (z.abs() >= z_threshold).astype(float)
    shock_cols = [col for col in out.columns if col.endswith('_shock')]
    out['any_external_shock'] = out[shock_cols].max(axis=1)
    return out


def summarize_binary_slice(df: pd.DataFrame, flag_col: str) -> pd.DataFrame:
    return (
        df.groupby(flag_col, dropna=False)
        .agg(
            rows=('market_id', 'size'),
            markets=('market_id', 'nunique'),
            repricing_rate=('target', 'mean'),
            mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
            mean_signed_future_move=('future_move', 'mean'),
        )
        .reset_index()
    )


## Load Polymarket Data and Build the Repricing Panel

We use the repricing dataset because the shock-propagation idea is fundamentally about short-horizon belief revision rather than terminal outcome prediction.

Each row in the resulting panel is a market snapshot at time `t`, labeled by whether the market reprices strongly over the next 24 hours.


In [ ]:
DATASET_ARTEFACT_DIR = REPO_ROOT / 'research_notebooks' / 'running_artefacts'

dataset = PolymarketDataset.from_parquet(DATASET_ARTEFACT_DIR)
markets = dataset.markets.copy()
probabilities = dataset.probabilities.copy()

prepared_markets = prepare_resolved_markets(markets)
prepared_markets = prepared_markets[prepared_markets['domain'].isin(DOMAINS)].copy()

snapshot_frame = load_snapshot_frame(probabilities, prepared_markets, horizon_hours=max(TERMINAL_HORIZONS))


## Build the External Shock Table

This cell constructs the external event series. For each timestamp we keep the raw return, its rolling z-score, and a binary shock indicator for `BTC/USD` and `ETH/USD`.

This gives us the event process that we will align with Polymarket repricing snapshots.


In [ ]:
shock_table = build_shock_table(EXTERNAL_PATH, z_threshold=SHOCK_Z_THRESHOLD, std_window=SHOCK_STD_WINDOW)
shock_summary = pd.DataFrame(
    {
        'series': ['btc_usd', 'eth_usd'],
        'shock_rate': [shock_table['btc_usd_shock'].mean(), shock_table['eth_usd_shock'].mean()],
        'max_abs_z': [shock_table['btc_usd_z'].abs().max(), shock_table['eth_usd_z'].abs().max()],
    }
)
display(shock_summary)
shock_table.head(3)


## Join Shocks to Repricing Snapshots

Now we attach the latest available external state to each Polymarket snapshot using an as-of join. This is the correct time-safe operation: each market row only sees external information available at that moment.

Once joined, we can ask whether repricing is more likely after external shocks and whether the effect varies by domain.


In [ ]:
repricing_with_shocks = asof_join_covariates(
    repricing,
    shock_table,
    base_time_col='timestamp_utc',
    covariate_time_col='timestamp_utc',
    max_age='2D',
)
repricing_with_shocks['btc_or_eth_shock'] = repricing_with_shocks[['btc_usd_shock', 'eth_usd_shock']].max(axis=1)

display(summarize_binary_slice(repricing_with_shocks, 'btc_usd_shock'))
display(summarize_binary_slice(repricing_with_shocks, 'eth_usd_shock'))
display(summarize_binary_slice(repricing_with_shocks, 'btc_or_eth_shock'))


## Domain-Level Response Patterns

Average effects can hide the main story. This cell breaks the response down by domain.

If the idea is real, we should see that some domains are more shock-sensitive than others, and that the effect shows up not only in repricing rate but also in the magnitude of future moves.


In [ ]:
domain_response = (
    repricing_with_shocks.groupby(['primary_domain', 'btc_or_eth_shock'], dropna=False)
    .agg(
        rows=('market_id', 'size'),
        repricing_rate=('target', 'mean'),
        mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
    )
    .reset_index()
)
display(domain_response.sort_values(['primary_domain', 'btc_or_eth_shock']))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=domain_response, x='repricing_rate', y='primary_domain', hue='btc_or_eth_shock', ax=axes[0], palette='crest')
axes[0].set_title('Repricing rate by domain and shock state')
axes[0].set_xlabel('Higher is more reactive')
axes[0].set_ylabel('')

sns.barplot(data=domain_response, x='mean_abs_future_move', y='primary_domain', hue='btc_or_eth_shock', ax=axes[1], palette='flare')
axes[1].set_title('Mean absolute future move by domain and shock state')
axes[1].set_xlabel('Higher means larger adjustment')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


## Response Distribution Under Shock vs Non-Shock States

This is a more distributional view of the same question. Instead of just comparing means, we compare the full distribution of future moves under shock and non-shock conditions.

This is helpful because a shock may not increase the repricing rate dramatically while still fattening the tail of future market moves.


In [ ]:
plot_df = repricing_with_shocks[['future_move', 'btc_or_eth_shock']].copy()
plot_df['shock_state'] = np.where(plot_df['btc_or_eth_shock'] >= 1.0, 'shock', 'no_shock')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.histplot(data=plot_df, x='future_move', hue='shock_state', bins=60, stat='density', common_norm=False, ax=axes[0])
axes[0].set_title('Future move distribution under shock vs no-shock')

sns.histplot(data=plot_df, x=np.abs(plot_df['future_move']), hue='shock_state', bins=60, stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Absolute future move distribution under shock vs no-shock')
axes[1].set_xlabel('|future_move|')
plt.tight_layout()
plt.show()


## Concrete Shock Examples

The notebook would be incomplete without examples. This cell surfaces actual markets that moved the most after shock-labeled snapshots.

These examples are useful for sanity checking whether the reactions look semantically plausible or whether the measured effect is mostly noise.


In [ ]:
example_cols = [
    'timestamp_utc', 'market_id', 'primary_domain', 'question',
    'current_yes_probability', 'future_move', 'target',
    'btc_usd_z', 'eth_usd_z', 'btc_usd_shock', 'eth_usd_shock'
]

btc_examples = (
    repricing_with_shocks.loc[repricing_with_shocks['btc_usd_shock'] >= 1.0, example_cols]
    .assign(abs_future_move=lambda x: x['future_move'].abs())
    .sort_values('abs_future_move', ascending=False)
    .head(15)
)
eth_examples = (
    repricing_with_shocks.loc[repricing_with_shocks['eth_usd_shock'] >= 1.0, example_cols]
    .assign(abs_future_move=lambda x: x['future_move'].abs())
    .sort_values('abs_future_move', ascending=False)
    .head(15)
)

print('Top reactions after BTC shocks')
display(btc_examples)
print('Top reactions after ETH shocks')
display(eth_examples)


## Shock-Centric View

Instead of ranking by market rows, we can also rank by shock timestamps and ask which windows produced the broadest response across Polymarket.

This is closer to an event-study view of the world: one external shock, many event-market reactions.


In [ ]:
shock_windows = (
    repricing_with_shocks.loc[repricing_with_shocks['btc_or_eth_shock'] >= 1.0]
    .groupby('timestamp_utc', dropna=False)
    .agg(
        affected_markets=('market_id', 'nunique'),
        repricing_rate=('target', 'mean'),
        mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
        max_btc_z=('btc_usd_z', lambda s: np.nanmax(np.abs(s))),
        max_eth_z=('eth_usd_z', lambda s: np.nanmax(np.abs(s))),
    )
    .reset_index()
    .sort_values(['repricing_rate', 'mean_abs_future_move'], ascending=False)
)
display(shock_windows.head(20))


## Reading the Feasibility Result

This notebook is useful if at least one of the following holds:

- repricing rates are materially higher after external shocks
- some domains are systematically more shock-sensitive than others
- the strongest examples look semantically plausible
- the effect shows up in future-move tails even if average differences are modest

If the signal is weak, the idea is probably not dead. It may simply mean that the right unit is not raw external shocks alone, but external shocks conditioned on domain, topic, or market closeness.
